# Sport and Health: Predicting Fitness Level from Lifestyle Data

**CPE393 – Introduction to Data Science with Python — Final Group Project**

**Group members:** Leni Chabilan · Romain Mechain · Kylian Riberou

---

## Project summary

This project studies how everyday lifestyle and activity factors relate to a
person's **fitness level**, using the *FitLife360* health & fitness dataset
(687,701 daily records from 3,000 participants tracked over the year 2024).

**Problem.** Rather than asking whether lifestyle predicts *disease* (which we
found this synthetic dataset does not support — see note below), we ask a
question the data *can* answer:

> **Which lifestyle and activity factors drive a person's fitness level, and
> how well can we predict that fitness level from their habits?**

This is a **regression** task. The target is `fitness_level`, a continuous
score (0.02 – 21.93).

**A note on methodology.** Because `fitness_level` varies from one exercise
session to the next, it is an inherently *daily* quantity, not a stable trait
of a person. We therefore model at the **daily record level**, but — crucially —
we split train/test **by participant** so that no individual appears in both
sets. This prevents data leakage, since the ~229 records of one person are not
independent of each other.


## 1. Setup and imports

We use Pandas/NumPy for data handling, Matplotlib/Seaborn for visualization,
and scikit-learn for modeling. A fixed `RANDOM_STATE` makes every result
reproducible.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Global settings for reproducibility and consistent figures
RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Loading the dataset

The dataset is the *FitLife360* synthetic health & fitness dataset.
Source (Kaggle): `jijagallery/fitlife-health-and-fitness-tracking-dataset`.

We first load the file and confirm its real size. The Kaggle description
mentions 3,000 participants; in fact the file contains one row **per
participant per day**, giving far more rows than participants.

In [2]:
# Load the raw data
df = pd.read_csv("health_fitness_dataset.csv")

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
df.head()

Rows:    687,701
Columns: 22


,participant_id,date,age,gender,height_cm,weight_kg,activity_type,duration_minutes,intensity,calories_burned,avg_heart_rate,hours_sleep,stress_level,daily_steps,hydration_level,bmi,resting_heart_rate,blood_pressure_systolic,blood_pressure_diastolic,health_condition,smoking_status,fitness_level
0,1,2024-01-01,56,F,165.3,53.7,Dancing,41,Low,3.3,103,6.6,3,7128,1.5,19.6,69.5,110.7,72.9,NaN,Never,0.04
1,1,2024-01-04,56,F,165.3,53.9,Swimming,28,Low,2.9,102,8.1,7,7925,1.8,19.6,69.5,110.7,72.9,NaN,Never,0.07
2,1,2024-01-05,56,F,165.3,54.2,Swimming,21,Medium,2.6,126,6.2,7,7557,2.7,19.6,69.5,110.7,72.9,NaN,Never,0.09
3,1,2024-01-07,56,F,165.3,54.4,Weight Training,99,Medium,10.7,141,7.2,8,11120,2.6,19.6,69.5,110.7,72.9,NaN,Never,0.21
4,1,2024-01-09,56,F,165.3,54.7,Swimming,100,Medium,12.7,112,7.1,1,5406,1.5,19.6,69.5,110.7,72.9,NaN,Never,0.33


## 3. Understanding the panel structure

This is **panel (longitudinal) data**: the same participants are observed
repeatedly over time. Each row is a *(participant, day)* pair. We confirm this
before doing anything else, because it dictates how we must split the data
later.

In [3]:
n_participants = df["participant_id"].nunique()
rows_per_participant = len(df) / n_participants

print(f"Unique participants : {n_participants:,}")
print(f"Date range          : {df['date'].min()} -> {df['date'].max()}")
print(f"Avg rows / person   : {rows_per_participant:.1f}")
print()
print("Records per participant (summary):")
print(df.groupby("participant_id").size().describe()[["min", "50%", "max"]].round(1))

Unique participants : 3,000
Date range          : 2024-01-01 -> 2024-12-25
Avg rows / person   : 229.2

Records per participant (summary):
min    198.0
50%    229.0
max    261.0
dtype: float64


## 4. Data quality audit

Before cleaning, we inventory the data types, missing values and duplicates so
that every cleaning decision in the next section is justified by evidence.

In [4]:
# Column types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 687701 entries, 0 to 687700
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   participant_id            687701 non-null  int64  
 1   date                      687701 non-null  str    
 2   age                       687701 non-null  int64  
 3   gender                    687701 non-null  str    
 4   height_cm                 687701 non-null  float64
 5   weight_kg                 687701 non-null  float64
 6   activity_type             687701 non-null  str    
 7   duration_minutes          687701 non-null  int64  
 8   intensity                 687701 non-null  str    
 9   calories_burned           687701 non-null  float64
 10  avg_heart_rate            687701 non-null  int64  
 11  hours_sleep               687701 non-null  float64
 12  stress_level              687701 non-null  int64  
 13  daily_steps               687701 non-null  int64  
 14 

In [5]:
# Missing values per column (only show columns that actually have any)
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)

quality = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
print("Columns with missing values:")
print(quality if not quality.empty else "None")
print(f"\nExact duplicate rows: {df.duplicated().sum()}")

Columns with missing values:
                  missing_count  missing_%
health_condition         490275       71.3



Exact duplicate rows: 0


**Reading the audit.**

* The only column with missing values is `health_condition` (~71% missing).
  This is **not** a data error: a missing value means the participant has *no
  recorded chronic condition* — i.e. they are healthy. We will encode this
  explicitly rather than drop those rows (dropping would delete most of the
  dataset).
* There are **no duplicate rows** and **no other missing values**, so the rest
  of the cleaning is light.

## 5. Cleaning and preprocessing

Each decision below is deliberate and explained:

1. **`date` → datetime.** Enables time-based EDA (seasonality, trends).
2. **`health_condition` missing → `"Healthy"`.** A missing value means no
   diagnosed condition. We make that explicit instead of dropping ~70% of rows.
3. **Type checks for categoricals.** Confirm the expected category sets so no
   stray/typo values slip through.
4. **Sanity ranges.** Verify numeric columns sit in plausible physiological
   ranges (no negative ages, BMIs, etc.).

In [6]:
# Work on a copy so the raw df stays available for comparison
data = df.copy()

# 1) Parse the date column to real datetime
data["date"] = pd.to_datetime(data["date"])

# 2) Missing health_condition means "no diagnosed condition" -> Healthy
data["health_condition"] = data["health_condition"].fillna("Healthy")

# 3) Confirm categorical columns only contain expected values
expected = {
    "gender":         {"M", "F", "Other"},
    "intensity":      {"Low", "Medium", "High"},
    "smoking_status": {"Never", "Former", "Current"},
}
for col, allowed in expected.items():
    found = set(data[col].unique())
    unexpected = found - allowed
    print(f"{col:15s} categories OK: {found}"
          + (f"  ⚠ UNEXPECTED: {unexpected}" if unexpected else ""))

print(f"\nhealth_condition categories: {sorted(data['health_condition'].unique())}")
print(f"activity_type categories   : {data['activity_type'].nunique()} types")

gender          categories OK: {'Other', 'F', 'M'}
intensity       categories OK: {'Medium', 'High', 'Low'}
smoking_status  categories OK: {'Never', 'Current', 'Former'}

health_condition categories: ['Asthma', 'Diabetes', 'Healthy', 'Hypertension']
activity_type categories   : 10 types


In [7]:
# 4) Sanity-check numeric ranges (look for impossible values)
numeric_cols = ["age", "height_cm", "weight_kg", "bmi", "duration_minutes",
                "calories_burned", "avg_heart_rate", "hours_sleep",
                "stress_level", "daily_steps", "hydration_level",
                "resting_heart_rate", "blood_pressure_systolic",
                "blood_pressure_diastolic", "fitness_level"]

ranges = data[numeric_cols].agg(["min", "max"]).T
ranges.columns = ["min", "max"]
print("Numeric column ranges:")
print(ranges.round(2))

# Explicit check: any negative values where they make no sense?
neg = (data[numeric_cols] < 0).sum()
print(f"\nColumns with negative values: "
      f"{list(neg[neg > 0].index) if neg.sum() else 'none'}")

Numeric column ranges:
                             min       max
age                        18.00     64.00
height_cm                 145.00    198.50
weight_kg                  45.30    188.40
bmi                        14.20     38.80
duration_minutes           20.00    120.00
calories_burned             0.80     92.00
avg_heart_rate             82.00    206.00
hours_sleep                 4.00     10.00
stress_level                1.00     10.00
daily_steps              -419.00  17241.00
hydration_level             1.50      3.50
resting_heart_rate         51.10     87.10
blood_pressure_systolic    78.00    152.70
blood_pressure_diastolic   53.70    112.10
fitness_level               0.02     21.93

Columns with negative values: ['daily_steps']


The range check surfaces one genuine anomaly: **`daily_steps` has a minimum
of −419**, which is physically impossible. Everything else is plausible. We
investigate and fix this below — exactly the kind of issue a data quality
audit is meant to catch.

In [8]:
# Investigate the impossible negative step counts
invalid_steps = data[data["daily_steps"] < 0]
print(f"Rows with negative daily_steps: {len(invalid_steps)} "
      f"({len(invalid_steps)/len(data)*100:.4f}% of data)")
print(f"Participants affected         : {invalid_steps['participant_id'].nunique()}")
print(invalid_steps[["participant_id", "date", "daily_steps"]].to_string(index=False))

Rows with negative daily_steps: 2 (0.0003% of data)
Participants affected         : 2
 participant_id       date  daily_steps
            905 2024-10-01         -419
           1573 2024-10-14          -81


Only **2 rows** out of 687,701 are affected. Rather than delete them (which
would also remove valid information in their other columns) or replace them
with 0 (which would invent a "no activity" day), we treat the negative values
as **invalid measurements**: we set them to `NaN` and impute each with that
participant's own median step count — the most representative substitute for
that individual.

In [9]:
# Replace impossible negative step counts with NaN, then impute per participant
data.loc[data["daily_steps"] < 0, "daily_steps"] = np.nan

# Impute each missing value with that participant's median daily_steps
data["daily_steps"] = data.groupby("participant_id")["daily_steps"].transform(
    lambda s: s.fillna(s.median())
)

# Confirm the fix
print(f"Negative step counts remaining: {(data['daily_steps'] < 0).sum()}")
print(f"Missing step counts remaining : {data['daily_steps'].isna().sum()}")
print(f"New daily_steps range          : "
      f"{data['daily_steps'].min():.0f} - {data['daily_steps'].max():.0f}")

Negative step counts remaining: 0
Missing step counts remaining : 0
New daily_steps range          : 36 - 17241


All categorical columns now contain only expected values, every numeric column
sits in a physiologically plausible range, the negative-step anomaly is fixed,
and there are no duplicates. The data is clean and ready for analysis.

## 6. Train/test split — *by participant* (avoiding data leakage)

This is the most important methodological step. Because each participant
contributes ~229 daily rows, a naive row-level split would place the **same
person** in both the training and test sets. The model could then "memorise"
individuals and report an over-optimistic score — a classic case of **data
leakage**, explicitly forbidden by the project's Responsible AI checklist.

To prevent this, we split on **`participant_id`**: 80% of *people* go to
training, 20% to testing, and all of a person's rows stay together on one side.

In [10]:
from sklearn.model_selection import train_test_split

# Split the PARTICIPANTS, not the rows
participant_ids = data["participant_id"].unique()

train_ids, test_ids = train_test_split(
    participant_ids, test_size=0.20, random_state=RANDOM_STATE
)

train_df = data[data["participant_id"].isin(train_ids)].copy()
test_df  = data[data["participant_id"].isin(test_ids)].copy()

print(f"Train: {len(train_df):>7,} rows  from {len(train_ids):,} participants")
print(f"Test : {len(test_df):>7,} rows  from {len(test_ids):,} participants")

# Verify there is NO participant overlap between train and test
overlap = set(train_ids) & set(test_ids)
print(f"\nParticipant overlap between train and test: {len(overlap)} "
      f"({'✓ no leakage' if not overlap else '⚠ LEAKAGE'})")

Train: 550,324 rows  from 2,400 participants
Test : 137,377 rows  from 600 participants

Participant overlap between train and test: 0 (✓ no leakage)


With zero participant overlap, any score we report later reflects how well the
model generalises to **people it has never seen** — exactly what we want.

---

### Foundations complete

We have loaded the data, confirmed its panel structure, audited and cleaned it
with justified decisions, and created a leakage-free participant-level split.
The next stages build on `train_df` / `test_df`:

* **Exploratory Data Analysis** (including the temporal dimension before we
  collapse it into features),
* **Feature engineering**,
* **Model training and comparison** (Linear Regression, Decision Tree,
  Random Forest),
* **Evaluation and Responsible AI discussion.**